## Colour statistics

In [ ]:
def calculate_channel_statistics(image_channel: np.ndarray) -> tuple[float, ...]:
    """
    Calculates comprehensive statistical features for an image channel.

    Args:
        image_channel: 2D numpy array representing a single image channel

    Returns:
        Tuple containing:
        - mean: Average pixel intensity
        - std_dev: Standard deviation of intensities
        - entropy: Information entropy of pixel distribution
        - kurt: Kurtosis (tailedness) of intensity distribution
        - energy: Total energy of the signal
        - skewness: Asymmetry of intensity distribution
    """
    # Flatten the 2D array to 1D for statistical calculations
    flat_channel = image_channel.ravel()

    # Intensity distribution analysis (32-bin histogram from 0-255)
    histogram_bins = np.arange(0, 256, 32)
    hist, _ = np.histogram(flat_channel, bins=histogram_bins)

    # Central tendency and dispersion
    mean = np.mean(flat_channel)
    std_dev = np.std(flat_channel)

    # Information theory measures
    hist_nonzero = hist[hist > 0]
    entropy_val = entropy(hist_nonzero) if hist_nonzero.size > 0 else np.nan

    # Shape characteristics
    kurt = kurtosis(flat_channel)
    skewness = skew(flat_channel)

    # Signal energy calculation
    energy = np.sum(flat_channel**2)

    return (mean, std_dev, entropy_val, kurt, energy, skewness)

In [ ]:
def extract_color_statistics_rgb(image_rgb: np.ndarray) -> tuple[tuple[float, ...], ...]:
    """
    Extracts statistical features for each channel of an RGB image.

    Args:
        image_rgb: Input image in RGB format with shape (H, W, 3)

    Returns:
        Tuple containing three feature tuples (red, green, blue channels)
        Each channel tuple contains:
        (mean, std_dev, entropy, kurtosis, energy, skewness)

    Raises:
        ValueError: If input is not a 3-channel RGB image

    """
    # Validate input dimensions
    if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
        raise ValueError("Input must be a 3-channel RGB image (H, W, 3)")

    # Extract color channels using numpy slicing
    red_channel = image_rgb[..., 0]  # More efficient than [:, :, 0]
    green_channel = image_rgb[..., 1]
    blue_channel = image_rgb[..., 2]

    # Calculate features for each channel
    return (
        calculate_channel_statistics(red_channel),
        calculate_channel_statistics(green_channel),
        calculate_channel_statistics(blue_channel)
    )

In [ ]:
def extract_color_statistics_hsv(image_rgb: np.ndarray) -> tuple[tuple[float, ...], ...]:
    """
    Extracts statistical features for each channel of an HSV image.

    Args:
        image_rgb: Input image in RGB format with shape (H, W, 3)

    Returns:
        Tuple containing three feature tuples (Hue, Saturation, Value)
        Each channel tuple contains:
        (mean, std_dev, entropy, kurtosis, energy, skewness)

    Raises:
        ValueError: If input is not a 3-channel RGB image


    Notes:
        - Hue range: 0-179 (OpenCV's HSV representation)
        - Saturation/Value range: 0-255
        - Requires calculate_channel_statistics() implementation
    """
    # Validate input dimensions
    if image_rgb.ndim != 3 or image_rgb.shape[2] != 3:
        raise ValueError("Input must be a 3-channel RGB image (H, W, 3)")

    # Convert to HSV color space (OpenCV expects BGR->HSV, ensure correct input)
    hsv_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)

    # Extract channels using efficient numpy slicing
    return (
        calculate_channel_statistics(hsv_image[..., 0]),  # Hue
        calculate_channel_statistics(hsv_image[..., 1]),  # Saturation
        calculate_channel_statistics(hsv_image[..., 2])   # Value
    )